In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
import pandas as pd
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint
from sklearn.decomposition import PCA

In [ ]:
def optimize_random_forest_with_PCA_toggle_with_scaler(
    X, y, pca_toggle=False, n_iter=10, n_PCs=45, random_state=4, printing=True
):
    """
    Optimize a Random Forest classifier with optional scaling and PCA.

    Inputs:
        X : pandas.DataFrame
            Feature matrix used for training and testing. Rows should be
            samples and columns should be features.

        y : pandas.Series or array-like
            Target labels for classification.

        pca_toggle : bool, default=False
            Whether to include PCA in the pipeline before fitting the
            Random Forest model.

        n_iter : int, default=10
            Number of parameter settings sampled in RandomizedSearchCV.

        n_PCs : int, default=45
            Number of principal components to keep if PCA is enabled.

        random_state : int, default=4
            Random seed used for the train/test split.

        printing : bool, default=True
            Whether to print the best parameters, cross-validation score,
            confusion matrix, classification report, and feature importances.

    Returns:
        random_search : RandomizedSearchCV
            Fitted RandomizedSearchCV object containing the best model and
            search results.

        X_test : pandas.DataFrame
            Test feature matrix used for final evaluation.

        y_test : pandas.Series or array-like
            True labels for the test set.

        feature_importances : pandas.DataFrame or None
            Feature importance table when PCA is not used. Returns None when
            PCA is enabled.
    """
    
    # Split the data into training and test sets
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=random_state, stratify=y
    )

    # Build pipeline: scaling first, optional PCA, then Random Forest
    steps = []
    steps.append(('scaler', StandardScaler()))
    
    if pca_toggle:
        steps.append(('pca', PCA()))
        
    steps.append(('rf', RandomForestClassifier(random_state=42)))
    pipeline = Pipeline(steps)

    # Define the hyperparameter search space for Random Forest
    hyperparameter_space = {
        'rf__n_estimators': randint(100, 500),
        'rf__max_depth': [None, 10, 20, 30, 40, 50],
        'rf__min_samples_split': randint(2, 10),
        'rf__min_samples_leaf': randint(1, 5)
    }

    # If PCA is enabled, keep the number of components fixed
    if pca_toggle:
        hyperparameter_space['pca__n_components'] = [n_PCs]

    # Tune the model with randomized search and 4-fold cross-validation
    random_search = RandomizedSearchCV(
        pipeline,
        param_distributions=hyperparameter_space,
        n_iter=n_iter,
        cv=4,
        n_jobs=-1,
        verbose=1,
        random_state=2,
        scoring='f1_macro'
    )
    
    # Fit the hyperparameter search on the training data
    random_search.fit(X_train, y_train)

    if printing:
        print(f"Best Parameters: {random_search.best_params_}")
        print(f"Best CV Score: {random_search.best_score_:.4f}")

    # Use the best fitted model to make predictions on the test set
    best_model = random_search.best_estimator_
    y_pred = best_model.predict(X_test)

    if printing:
        print("\nConfusion Matrix:")
        print(confusion_matrix(y_test, y_pred))
        print("\nClassification Report:")
        print(classification_report(y_test, y_pred))

    # Extract feature importances only when PCA is not used
    if not pca_toggle:
        rf_model = best_model.named_steps['rf']
        feature_importances = pd.DataFrame({
            'Feature': X.columns,
            'Importance': rf_model.feature_importances_
        }).sort_values(by='Importance', ascending=False)

        if printing:
            print("\nTop 10 Most Important Features:")
            print(feature_importances.head(10))

    else:
        if printing:
            print("\nPCA enabled - feature importances are not available.")
        feature_importances = None

    return random_search, X_test, y_test, feature_importances


In [ ]:
from sklearn.metrics import confusion_matrix, f1_score, accuracy_score, recall_score

def repeat_rf_optimization(
    X, y, 
    n_runs=50, 
    pca_toggle=False, 
    n_iter=20, 
    n_PCs=25
):
    """
    Repeatedly run Random Forest optimization and summarize model performance
    across multiple random train/test splits.

    Inputs:
        X : pandas.DataFrame
            Feature matrix with samples as rows and features as columns.

        y : pandas.Series or array-like
            Target labels for classification.

        n_runs : int, default=50
            Number of repeated runs to perform with different random states.

        pca_toggle : bool, default=False
            Whether PCA is included in the optimization pipeline.

        n_iter : int, default=20
            Number of parameter settings sampled in RandomizedSearchCV.

        n_PCs : int, default=25
            Number of principal components to keep if PCA is enabled.

    Returns:
        confusions : list
            List of confusion matrices, one per run.

        f1_scores : list
            List of macro F1 scores across runs.

        accuracies : list
            List of accuracy values across runs.

        recalls : list
            List of macro recall values across runs.

        avg_confusion : numpy.ndarray
            Mean confusion matrix across all runs.

        avg_importances : pandas.DataFrame or None
            Average feature importance across runs if PCA is disabled;
            otherwise None.
    """

    confusions = []
    f1_scores = []
    accuracies = []
    recalls = []
    importances_list = []   # only used when PCA is off

    # Repeat the optimization several times with different random splits
    for i in range(n_runs):

        print(f"Run {i+1}/{n_runs}\n")

        rs, X_test, y_test, feature_importances = optimize_random_forest_with_PCA_toggle_with_scaler(
            X, y, pca_toggle=pca_toggle, n_iter=n_iter, n_PCs=n_PCs, random_state=i + 42
        )

        # Use the best model found during hyperparameter search
        best_model = rs.best_estimator_
        y_pred = best_model.predict(X_test)

        # Store confusion matrix for this run
        cm = confusion_matrix(y_test, y_pred)
        confusions.append(cm)

        # Compute and store evaluation metrics
        f1 = f1_score(y_test, y_pred, average="macro")
        acc = accuracy_score(y_test, y_pred)
        rec = recall_score(y_test, y_pred, average="macro")

        f1_scores.append(f1)
        accuracies.append(acc)
        recalls.append(rec)

        # Store feature importances only when PCA is not used
        if not pca_toggle:
            importances_list.append(
                feature_importances.set_index('Feature').loc[X.columns]['Importance'].values
            )

    # Average confusion matrix across runs
    avg_confusion = np.mean(confusions, axis=0)

    print("\n=======================================")
    print("Average Confusion Matrix Across Runs")
    print("=======================================")
    print(avg_confusion)

    print("\n=======================================")
    print("Performance Summary")
    print("=======================================")
    print(f"Mean Accuracy:      {np.mean(accuracies):.4f}")
    print(f"Std  Accuracy:      {np.std(accuracies):.4f}")
    print(f"Mean Recall-macro:  {np.mean(recalls):.4f}")
    print(f"Std  Recall-macro:  {np.std(recalls):.4f}")
    print(f"Mean F1-macro:      {np.mean(f1_scores):.4f}")
    print(f"Std  F1-macro:      {np.std(f1_scores):.4f}")

    # Average feature importance across runs if PCA is disabled
    if not pca_toggle:
        avg_importance_values = np.mean(importances_list, axis=0)
        avg_importances = pd.DataFrame({
            'Feature': X.columns,
            'AvgImportance': avg_importance_values
        }).sort_values(by='AvgImportance', ascending=False)

        print("\n=======================================")
        print("Average Feature Importance Across Runs")
        print("=======================================")
        print(avg_importances.head(20))

    else:
        avg_importances = None
        print("\nPCA enabled — skipping feature importance aggregation.")

    return confusions, f1_scores, accuracies, recalls, avg_confusion, avg_importances

In [ ]:
def get_condition(data_with_conditions, row_name='Diet'):
    '''
    Extract diet information from the metadata file.
    
    Inputs:
        data_with_conditions : str
            Path to the metadata CSV file containing 'Diet' and 'OmicsEarTag' rows.
        row : int (unused, kept for backward compatibility)
            Placeholder parameter (can be ignored).
    
    Outputs:
        pd.DataFrame - Clean mapping of FAC_ID to Diet
    '''
    # Load metadata
    meta = pd.read_csv(data_with_conditions, index_col=0)
    
    # Extract relevant rows 
    diet_row = meta.loc[row_name]
    fac_row = meta.loc['OmicsEarTag']
    
    # Build mapping
    diet_map = pd.DataFrame({
        'FAC_ID': fac_row.values,
        row_name: diet_row.values
    })
    
    # Drop unwanted rows (header-like entries)
    diet_map = diet_map[~diet_map['FAC_ID'].isin(['Orig_ID', 'IDFemaleAgingColony'])].reset_index(drop=True)
    
    return diet_map

In [ ]:
def prep_data_for_RF_omics(transcriptomics_diet, condition_row="Diet"):
    """
    Prepare transcriptomics data for Random Forest classification by separating features and labels.
    Inputs:
        transcriptomics_diet: DataFrame with one row containing Diet labels (HF/CD)
        condition_row: name of that row (default = 'Diet')
    Returns:
        X: DataFrame of transcript features (samples as rows, features as columns)
        y: Series of Diet labels (indexed by sample)
    """

    # Ensure index has a name
    if transcriptomics_diet.index.name is None:
        transcriptomics_diet.index.name = "Row"

    # Extract y = Diet labels
    y = transcriptomics_diet.loc[condition_row]

    # Remove NaNs from y
    y = y.dropna()

    # Extract X = transcript rows (remove Diet row)
    X = transcriptomics_diet.drop(index=condition_row)

    # Remove non-numeric column if present
    if "Orig_ID" in X.columns:
        X = X.drop(columns=["Orig_ID"])

    # Filter X to keep only columns that exist in cleaned y
    X = X[y.index]

    # Convert to numeric
    X = X.apply(pd.to_numeric, errors="coerce")

    print("X shape:", X.shape)
    print("y shape:", y.shape)
    print("\ny value counts:")
    print(y.value_counts())
    
    return X, y

In [ ]:
transcriptomics = pd.read_csv('../Transcriptomics_datasets/process_trans_file_1_col.csv') # path to transcriptomics data with one row of Diet labels

In [ ]:
diet_data = get_condition("../aData_S1_AllOmicsandPhenotypeData.csv") # get diet information from metadata file

In [ ]:
transcriptomics

In [ ]:
diet_data

In [ ]:
# Drop the ID column so only FAC columns remain
fac_cols = [c for c in transcriptomics.columns if c.startswith("FAC")]

diet_row = diet_data.set_index("FAC_ID").loc[fac_cols].T

# Add a row labeled "Diet"
transcriptomics_diet = pd.concat([diet_row, transcriptomics], ignore_index=False)

# First move the Orig_ID column into the index
transcriptomics_diet = transcriptomics_diet.set_index("Orig_ID")

# Rename the index entry for the diet row
transcriptomics_diet = transcriptomics_diet.rename(index={np.nan: "Diet"})

transcriptomics_diet

In [ ]:
X, y = prep_data_for_RF_omics(transcriptomics_diet, "Diet") # prepare data for Random Forest classification
X = X.T

In [ ]:
X

In [ ]:
transcriptomics_results = repeat_rf_optimization(X, y, n_runs=30, pca_toggle=False, n_iter=10) # run repeated Random Forest optimization without PCA

In [ ]:
df = pd.DataFrame(np.vstack(transcriptomics_results[0])) # transcriptomics_results[0] contains the list of confusion matrices, we stack them vertically and convert to DataFrame
df.to_csv("../final_result/transcriptomics_results_diet.csv", index=False)